In [ ]:
from tqdm import tqdm
from port.query_structure import GenerateQueryStructure

import pandas as pd
import numpy as np
import config
import pickle
import re
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def extract_name(data):
    match = re.search(r'\((.*?)\)', data)
    return match.group(1) if match else data

def compute_score(data):

    x = data.sort_values(ascending=True)
    max_val = len(x)

    if max_val <= 10:
        return np.nan
    else:
        return (x - 1) * (99 / (max_val - 1)) + 1

def assign_quantile(x):
    if 1 <= x <= 20:
        return 'Q1'
    elif 21 <= x <= 40:
        return 'Q2'
    elif 41 <= x <= 60:
        return 'Q3'
    elif 61 <= x <= 80:
        return 'Q4'
    elif 81 <= x <= 100:
        return 'Q5'
    else:
        return 'Out_of_range'
    
def calc_factor_returns(result_query, factor_name, data, ascending_order):

    result_fld = data[data['fld'] == factor_name].reset_index(drop=True)
    result_fld['lval'] = result_fld.groupby('gvkeyiid')['val'].shift(1)
    result_fld = result_fld.rename(columns={'lval': result_fld['fld'].iloc[0]})
    result_fld = result_fld.dropna(subset=[result_fld['fld'].iloc[0]])
    result_fld = result_fld.drop(columns=['val', 'fld'])

    if len(result_fld) <= 100:
        print(factor_name)
        return None, None, None, None

    else:

        result_rtn = result_query[result_query['fld'] == 'M_RETURN'].reset_index(drop=True)
        result_rtn = result_rtn.rename(columns={'val': result_rtn['fld'].iloc[0]})
        result_rtn = result_rtn.drop(columns=['fld'])

        result_df = pd.merge(result_fld, result_rtn, on=['gvkeyiid', 'ticker', 'isin', 'ddt', 'sec', 'country'], how='inner')
        result_df = result_df[result_df['sec'] != 'Undefined'].reset_index(drop=True)

        result_df['rank'] = result_df.groupby(['ddt', 'sec'])[result_df.columns[11]].rank(method='average', ascending=ascending_order)
        result_df['score'] = result_df.groupby(['ddt', 'sec'])['rank'].transform(compute_score)
        result_df['score'] = np.round(result_df['score'], 1)
        result_df['quantile'] = result_df['score'].apply(assign_quantile)

        result_sec = result_df.groupby(['ddt', 'sec', 'quantile'])['M_RETURN'].mean().unstack()
        result_quantile = result_df.groupby(['ddt', 'quantile'])['M_RETURN'].mean().unstack()
        
        if set('Out_of_range').issubset(result_sec.columns):

            result_sec = result_sec.drop(columns=['Out_of_range'])
            result_quantile = result_quantile.drop(columns=['Out_of_range'])
        else:
            pass

        result_sec = result_sec.fillna(0)
        result_sec = result_sec.groupby('sec').mean().T * 100

        result_quantile = result_quantile.fillna(0)

        result_spread = pd.DataFrame({f'{factor_name}': result_quantile.iloc[:,1] - result_quantile.iloc[:,-1]})

        first_date = result_spread.index[0]
        init_date = first_date - pd.DateOffset(months=1)
        init_df = pd.DataFrame({f'{factor_name}': [0.0]}, index=[init_date])

        result_spread = pd.concat([result_spread, init_df], axis=0).sort_index(ascending=True)

        return result_sec, result_quantile, result_spread, result_df


In [ ]:
print("Welcome to S&P Global Capital IQ")
query_structure = GenerateQueryStructure()
result_query = query_structure.fetch_snp()

result_query['fld'] = result_query['fld'].apply(extract_name)

result_dd = result_query.drop_duplicates(subset=['ddt', 'gvkeyiid', 'fld', 'val'], keep='last', ignore_index=True)
result_info = pd.read_csv("data/factor_master.csv")
result_info = result_info.rename(columns={'factorAbbreviation':'fld'})

result_master = pd.merge(result_dd, result_info, on='fld', how='inner')

_unique_master = result_master.drop_duplicates(subset=['fld'])[['fld', 'factorShortName', 'factorOrder']]
_unique_master = _unique_master.rename(columns={'fld': 'factor_name'})

unique_master = pd.read_csv("data/factor_info.csv", index_col=0)
unique_master = pd.merge(_unique_master, unique_master, on='factor_name', how='inner')
unique_master = unique_master.sort_values('rank').reset_index(drop=True)
unique_master = unique_master[['cagr', 'rank', 'factor_name', 'factorShortName', 'factorName', 'styleName', 'factorOrder']]
unique_master = unique_master.rename(columns={'factor_name': 'factorAbbreviation'})

unique_master = result_master.drop_duplicates(subset=['fld'])
unique_name = unique_master.fld.tolist()

unique_master.loc[unique_master['factorOrder'] == 'Descending', 'factorOrder'] = False
unique_master.loc[unique_master['factorOrder'] == 'Ascending', 'factorOrder'] = True
unique_order = unique_master.factorOrder.tolist()

list_sector, list_quantile, list_returns, list_raw = [], [], [], []

for i in tqdm(range(0, len(unique_name))):
    if ((unique_name[i] == 'MXCN1A_WGT') or (unique_name[i] == 'M_RETURN')):
        print("Not Allowed")
        pass
    else:
        res_sector, res_quantile, res_return, res_raw = calc_factor_returns(result_query, unique_name[i], result_master, unique_order[i])
        list_sector.append(res_sector)
        list_quantile.append(res_quantile)
        list_returns.append(res_return)
        list_raw.append(res_raw)

df = pd.concat([s for s in list_returns], axis=1)
df = df.fillna(0)

Welcome to S&P Global Capital IQ
Database Connected


FileNotFoundError: [Errno 2] No such file or directory: 'factor_master.csv'

In [ ]:
with open("data/list_quantile.pkl", "wb") as f:
    pickle.dump(list_quantile, f)

with open("data/list_factor_name.pkl", "wb") as f:
    pickle.dump(unique_name, f)

with open("data/list_full_name.pkl", "wb") as f:
    pickle.dump(unique_master['factorName'].tolist(), f)

with open("data/list_sector.pkl", "wb") as f:
    pickle.dump(list_sector, f)

with open("data/list_returns.pkl", "wb") as f:
    pickle.dump(list_returns, f)

with open("data/list_raw.pkl", "wb") as f:
    pickle.dump(list_raw, f)

with open("data/list_style_name.pkl", "wb") as f:
    pickle.dump(unique_master['styleName'].tolist(), f)

df.to_csv("data/result_factor_rtn.csv")